# Image Quality: Seeing Histograms and Healpix Maps

Comparison of the delivered image quality (seeing) between Rubin/LSST feature-scheduler
cadence simulations, using `rubin_sim.maf`.

- created: 2026-09-17
- kernel: conda_py313_opsim53
- Purpose: characterize the per-visit seeing (`seeingFwhmEff`, `seeingFwhmGeom`) delivered by
  different cadence simulations, as Healpix maps (`nside=64`) and all-sky histograms, and
  compare consecutive baseline versions (starting with `v5.3.6` vs `v5.3.5`, falling back to
  older baselines if `v5.3.5` is not available on disk).
- `seeingFwhmEff`: effective seeing used for point-source photometric SNR (what matters for
  depth / detection).
- `seeingFwhmGeom`: geometric/physical seeing, used e.g. for astrometric precision and shape
  measurements (what matters for weak lensing / strong lensing image quality).
  Both are per-visit, airmass-corrected values already stored in the opsim database, so a
  Healpix median map reflects both the atmospheric seeing model *and* the airmass distribution
  of visits on the sky (fields near the horizon accumulate worse seeing).
- References:
  - `rubin_sim.maf` source: https://github.com/lsst/rubin_sim
  - Column definitions: https://rubin-sim.lsst.io (opsim `observations` schema)
  - Follows the repository convention (`data_<NN_TAG>/` for MAF outputs, `figs_<NN_TAG>/` for
    figures, saved as PNG + PDF), as established in `../06_MAF_DESC_TaskF/` and
    `../07_variateEVmV/`.
  - Builds on the `CountMetric`/`HealpixSlicer` pattern from `../03_fbs5.3.6/Footprint.ipynb`.

In [ ]:
# import necessary and useful packages
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
import rubin_sim.maf as maf

## 1. Locate simulations to compare

Auto-discover which `baseline_v5.3.*` databases are actually present on disk, rather than hard-coding a filename that may not exist locally.

In [ ]:
path_topsim = os.getenv("RUBIN_SIM_DATA_DIR")
print("RUBIN_SIM_DATA_DIR =", path_topsim)

# Baselines are usually stored under sim_baseline/, but search a couple of likely locations
search_dirs = [
    os.path.join(path_topsim, "sim_baseline"),
    path_topsim,
]

candidates = {}
for d in search_dirs:
    if d and os.path.isdir(d):
        for f in glob.glob(os.path.join(d, "baseline_v*_10yrs.db")):
            run_name = os.path.split(f)[-1].replace(".db", "")
            candidates[run_name] = f

# Also allow a plain 'baseline.db' or other naming used by some releases
for d in search_dirs:
    if d and os.path.isdir(d):
        for f in glob.glob(os.path.join(d, "baseline*.db")):
            run_name = os.path.split(f)[-1].replace(".db", "")
            candidates.setdefault(run_name, f)

for run_name, f in sorted(candidates.items()):
    print(f"{run_name:35s} -> {f}")

In [ ]:
# Pick which runs to actually compare: v5.3.6 (primary), then v5.3.5 if present,
# else fall back to the next older baseline found on disk. Edit `wanted` by hand if you
# want a different / longer comparison list once you've seen what `candidates` contains above.

wanted = ["baseline_v5.3.6_10yrs", "baseline_v5.3.5_10yrs"]
runs = [r for r in wanted if r in candidates]

if len(runs) < 2:
    # fall back: v5.3.6 (or the newest found) plus the next-older baseline available
    all_found = sorted(candidates.keys())
    if "baseline_v5.3.6_10yrs" in candidates:
        runs = ["baseline_v5.3.6_10yrs"]
    elif all_found:
        runs = [all_found[-1]]
    else:
        runs = []
    older = [r for r in all_found if r not in runs]
    if older:
        runs.append(older[-1])

print("Runs selected for this notebook:", runs)
assert (
    len(runs) >= 1
), "No baseline_v*.db files found under RUBIN_SIM_DATA_DIR -- check the path / download the data."

run_to_db = {r: candidates[r] for r in runs}
run_to_db

## 2. Output directories (repository convention)

In [ ]:
TAG = "01_Seeing"
data_dir = f"data_{TAG}"
figs_dir = f"figs_{TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)


def savefig(fig, name):
    fig.savefig(os.path.join(figs_dir, name + ".png"), format="png", bbox_inches="tight", dpi=150)
    fig.savefig(os.path.join(figs_dir, name + ".pdf"), format="pdf", bbox_inches="tight")

## 3. Run the seeing metrics (Healpix map + histogram)

For each simulation: median `seeingFwhmEff` and median `seeingFwhmGeom` over all visits, all
bands, `nside=64` -- the same slicer resolution used for the `fO`/`NVisits` maps in
`03_fbs5.3.6/Footprint.ipynb`.

In [ ]:
nside = 64
seeing_cols = {
    "seeingFwhmEff": "Effective seeing (point-source SNR)",
    "seeingFwhmGeom": "Geometric seeing (astrometry / shapes)",
}

bundles = {run_name: {} for run_name in runs}

for run_name in runs:
    opsdb = run_to_db[run_name]
    bundle_dict = {}
    for col in seeing_cols:
        metric = maf.MedianMetric(col, metric_name=f"Median {col}")
        slicer = maf.HealpixSlicer(nside=nside)
        constraint = None
        plot_dict = {"x_min": 0.3, "x_max": 2.0, "color_min": 0.3, "color_max": 2.0}
        plot_funcs = [maf.HealpixSkyMap(), maf.HealpixHistogram()]
        b = maf.MetricBundle(
            metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
        )
        bundle_dict[f"{run_name}_{col}"] = b

    g = maf.MetricBundleGroup(bundle_dict, opsdb, out_dir=data_dir, verbose=True)
    g.run_all()

    for col in seeing_cols:
        bundles[run_name][col] = bundle_dict[f"{run_name}_{col}"]

print("Done:", [f"{r}/{c}" for r in runs for c in seeing_cols])

### 3.1 Sky maps + histograms, one run at a time

In [ ]:
for run_name in runs:
    for col, label in seeing_cols.items():
        b = bundles[run_name][col]
        b.set_plot_dict(
            {"x_min": 0.3, "x_max": 2.0, "color_min": 0.3, "color_max": 2.0, "xlabel": f"{label} (arcsec)"}
        )
        figs = b.plot()
        # b.plot() draws into the current figure(s); grab and save the last-created figure(s)
        for num in plt.get_fignums():
            fig = plt.figure(num)
            savefig(fig, f"{run_name}_{col}")
        plt.show()

## 4. Side-by-side comparison, common color scale

All selected runs plotted on the same `mollview` color range, for each seeing column.

In [ ]:
for col, label in seeing_cols.items():
    n = len(runs)
    fig = plt.figure(figsize=(7 * n, 5))
    for i, run_name in enumerate(runs):
        mval = bundles[run_name][col].metric_values.filled(hp.UNSEEN)
        hp.mollview(
            mval,
            min=0.3,
            max=2.0,
            title=f"{run_name}\n{label}",
            unit="arcsec",
            sub=(1, n, i + 1),
            fig=fig.number,
        )
    savefig(fig, f"mosaic_{col}")
    plt.show()

## 5. Difference maps between consecutive baselines

`runs[0]` is taken as the more recent version, `runs[1]` (if present) as the reference/older
one: `runs[0] - runs[1]`, per Healpix pixel.

In [ ]:
if len(runs) >= 2:
    newer, older = runs[0], runs[1]
    for col, label in seeing_cols.items():
        v_new = bundles[newer][col].metric_values.filled(np.nan)
        v_old = bundles[older][col].metric_values.filled(np.nan)
        diff = v_new - v_old

        fig = plt.figure(figsize=(7, 5))
        hp.mollview(
            diff,
            min=-0.2,
            max=0.2,
            cmap="RdBu_r",
            title=f"{label}: {newer} - {older}",
            unit="arcsec",
            fig=fig.number,
        )
        savefig(fig, f"diff_{col}_{newer}_minus_{older}")
        plt.show()

        finite = diff[np.isfinite(diff)]
        print(
            f"{col}: mean diff = {np.nanmean(finite):.4f} arcsec, "
            f"median diff = {np.nanmedian(finite):.4f} arcsec, std = {np.nanstd(finite):.4f} arcsec"
        )
else:
    print(
        "Only one run available -- skipping difference maps. Add a second baseline .db to RUBIN_SIM_DATA_DIR to enable this section."
    )

## 6. Overlaid all-sky histograms (direct comparison)

Pixel-value distributions for all selected runs on the same axes, for each seeing column.

In [ ]:
for col, label in seeing_cols.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    for run_name in runs:
        vals = bundles[run_name][col].metric_values.compressed()
        ax.hist(vals, bins=80, range=(0.3, 2.0), histtype="step", linewidth=2, label=run_name, density=True)
    ax.set_xlabel(f"{label} (arcsec)", fontsize="large")
    ax.set_ylabel("Normalized pixel count", fontsize="large")
    ax.legend()
    ax.grid(True, alpha=0.3)
    savefig(fig, f"hist_overlay_{col}")
    plt.show()

## 7. Per-band breakdown

Median seeing per band, per run, all-sky (`HealpixSlicer`, one map per band, summarized to a single number each).

In [ ]:
bands = "ugrizy"
per_band_rows = []

for run_name in runs:
    opsdb = run_to_db[run_name]
    for col in seeing_cols:
        bundle_dict = {}
        for band in bands:
            metric = maf.MedianMetric(col, metric_name=f"Median {col} {band}")
            slicer = maf.HealpixSlicer(nside=nside)
            constraint = f"filter = '{band}'"
            b = maf.MetricBundle(metric, slicer, constraint, run_name=run_name)
            bundle_dict[band] = b
        g = maf.MetricBundleGroup(bundle_dict, opsdb, out_dir=data_dir, verbose=False)
        try:
            g.run_all()
        except Exception:
            # some newer sims use 'band' instead of 'filter' in the sql constraint
            bundle_dict = {}
            for band in bands:
                metric = maf.MedianMetric(col, metric_name=f"Median {col} {band}")
                slicer = maf.HealpixSlicer(nside=nside)
                constraint = f"band = '{band}'"
                b = maf.MetricBundle(metric, slicer, constraint, run_name=run_name)
                bundle_dict[band] = b
            g = maf.MetricBundleGroup(bundle_dict, opsdb, out_dir=data_dir, verbose=False)
            g.run_all()

        for band in bands:
            vals = bundle_dict[band].metric_values.compressed()
            if len(vals):
                per_band_rows.append(
                    dict(run=run_name, metric=col, band=band, median_all_sky=float(np.median(vals)))
                )

per_band_df = pd.DataFrame(per_band_rows)
per_band_df

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 5), sharey=False)
for ax, col in zip(axes, seeing_cols):
    sub = per_band_df.query("metric == @col")
    for run_name in runs:
        s = sub.query("run == @run_name").set_index("band").reindex(list(bands))
        ax.plot(list(bands), s["median_all_sky"], marker="o", label=run_name)
    ax.set_title(seeing_cols[col])
    ax.set_xlabel("band")
    ax.set_ylabel("Median all-sky seeing (arcsec)")
    ax.grid(True, alpha=0.3)
    ax.legend()
savefig(fig, "per_band_comparison")
plt.show()

## 8. Summary table (all-sky statistics)

In [ ]:
summary_rows = []
for run_name in runs:
    for col, label in seeing_cols.items():
        vals = bundles[run_name][col].metric_values.compressed()
        summary_rows.append(
            dict(
                run=run_name,
                metric=col,
                mean=float(np.mean(vals)),
                median=float(np.median(vals)),
                std=float(np.std(vals)),
                min=float(np.min(vals)),
                max=float(np.max(vals)),
            )
        )

summary_df = pd.DataFrame(summary_rows).set_index(["run", "metric"]).round(4)
summary_df.to_csv(os.path.join(data_dir, "seeing_summary_stats.csv"))
summary_df

## Notes / next steps

- To extend the comparison to more than two runs, add more entries to `wanted` in section 1
  (any `baseline_v*_10yrs.db`, or the `shrink_fp_dust_*` / other cadence family runs following
  the pattern used in `../07_variateEVmV/`).
- `seeingFwhmEff` and `seeingFwhmGeom` are airmass-corrected per-visit values already present in
  the opsim database -- no extra stacker is needed to compute them.
- For a scalar, single-number-per-run comparison (rather than full maps), the pre-computed
  `summary.h5` files under `RUBIN_SIM_DATA_DIR/maf/<fbs_version>/summary.h5` may already contain
  seeing-related summary statistics (look for columns containing `"seeingFwhmEff"` or
  `"seeingFwhmGeom"`), following the pattern used for `WFD Depths` in
  `../03_fbs5.3.6/Footprint.ipynb`. This is only available for versions the LSST team has
  batch-processed, which may not include arbitrary older baselines.